# 06 Tribal-Managed Soil Profiles

This notebook analyzes only explicitly authorized field records. Public SSURGO horizons are not expected for Pine Ridge and are not a prerequisite. Empty authorized input is a valid outcome, not an invitation to substitute surrounding-county data.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"src").is_dir())
sys.path.insert(0, str(REPO_ROOT)) if str(REPO_ROOT) not in sys.path else None
import pandas as pd
import geopandas as gpd
import yaml
from IPython.display import display
from src.constants import REPO_ROOT as ROOT, OUTPUTS_DIR
from src.loaders import load_tribal_boundaries
with open(ROOT/"config"/"config.yaml") as stream: CONFIG = yaml.safe_load(stream)
primary = load_tribal_boundaries(["Pine Ridge", "Rosebud"])
pine_ridge = primary[primary["NAME"] == "Pine Ridge"]
rosebud = primary[primary["NAME"] == "Rosebud"]

## Governance and authorization gate

Required record fields are `data_authority`, `access_level`, `authorized_use`, and `authorization_date`. The template example row is illustrative and is never loaded automatically.

In [2]:
from src.loaders import load_tribal_soil_profiles
from src.soil_evidence import validate_governed_profiles, GovernanceError, REQUIRED_GOVERNANCE_FIELDS
profiles = load_tribal_soil_profiles()
print(f"Records discovered in governed raw-data location: {len(profiles):,}")
try:
    authorized_profiles = validate_governed_profiles(profiles, authorized_use="soils analysis")
except GovernanceError as exc:
    authorized_profiles = pd.DataFrame()
    print(f"Governance gate stopped analysis: {exc}")
print(f"Records authorized for this use: {len(authorized_profiles):,}")

Records discovered in governed raw-data location: 0
Records authorized for this use: 0


C:\Users\gekek\AppData\Local\Temp\ipykernel_3456\3822589369.py:3: UserWarning: No Tribal soil profile data found in data/raw/. See Field data forms/soil_profile_template.xlsx to begin collecting field measurements.
  profiles = load_tribal_soil_profiles()


## Field-data quality checks

Checks are applied only after authorization. They validate identifiers, coordinates, horizon depth order, overlap, and plausible field ranges without altering raw observations.

In [3]:
if authorized_profiles.empty:
    print("No authorized profiles: soil properties remain UNKNOWN. No maps or summaries are produced.")
else:
    q = authorized_profiles.copy()
    for column in ["lat", "lon", "depth_top_cm", "depth_bottom_cm", "ph"]:
        q[column] = pd.to_numeric(q[column], errors="coerce")
    q["valid_coordinates"] = q["lat"].between(42, 46) & q["lon"].between(-105, -96)
    q["valid_depth_order"] = q["depth_top_cm"].ge(0) & q["depth_bottom_cm"].gt(q["depth_top_cm"])
    q["valid_ph"] = q["ph"].isna() | q["ph"].between(0, 14)
    q["quality_pass"] = q[["valid_coordinates", "valid_depth_order", "valid_ph"]].all(axis=1)
    display(q[["profile_id", "horizon", "quality_pass", "valid_coordinates", "valid_depth_order", "valid_ph"]])
    analysis_profiles = q[q["quality_pass"]].copy()
    print(f"Authorized records passing quality checks: {len(analysis_profiles):,}")

No authorized profiles: soil properties remain UNKNOWN. No maps or summaries are produced.


## Evidence limit

Results from authorized profiles describe sampled locations only. They are not reservation-wide estimates unless a separately reviewed sampling design supports that inference.